## ML template

### Step 1: Preprocessing the data:
* import dependencies,
* separate target variable from explanatory features
* clean the data: remove highly correlated features, remove outliers (or not), etc. feature transformation: --> X is skewed X>0, log(X) gets more bell shaped
* split the data and dummify the categorical features
* standardize the features

### Step 2: building the model:
* fit the model in the train data

### Step 3: Evaluate the model:
* choose a performance metric that aligns with our goals
* calculate the predictions
* evaluate the performance in the train/test set

### Step 4: Explainability:
* feature selection to understand the individual contribution of each feature to the overall predictions

In [15]:
##import

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder ## it is like dummification but does not drop the column n by default
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Salary_Data.csv to Salary_Data.csv


In [4]:
df = pd.read_csv('Salary_Data.csv')
df

,Country,YearsExperience,Salary
0,France,1.1,39343.0
1,United-Kingdom,1.3,46205.0
2,France,1.5,37731.0
3,France,2.0,43525.0
4,Germany,2.2,39891.0
5,United-Kingdom,2.9,56642.0
6,United-Kingdom,3.0,60150.0
7,Germany,3.2,54445.0
8,Germany,3.2,64445.0
9,Germany,3.7,57189.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Country          30 non-null     object 
 1   YearsExperience  30 non-null     float64
 2   Salary           30 non-null     float64
dtypes: float64(2), object(1)
memory usage: 852.0+ bytes


In [6]:
## eda, clean outliers, missing nans

numerical_data = df.select_dtypes(include=['int64', 'float64'])
numerical_data

categorical = df.select_dtypes(include=['object'])
categorical


,Country
0,France
1,United-Kingdom
2,France
3,France
4,Germany
5,United-Kingdom
6,United-Kingdom
7,Germany
8,Germany
9,Germany


In [7]:
X = df.drop(columns ="Salary")
y = df["Salary"]

In [8]:
X.Country.value_counts()

,count
Country,
France,12
United-Kingdom,12
Germany,6


In [21]:
## train test split

## strategy: test data is just regarding Germany
X_train = X.loc[X.Country != 'Germany']
y_train = y.loc[X.Country != 'Germany']

X_test = X.loc[X.Country == 'Germany']
y_test = y.loc[X.Country == 'Germany']

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
display(X_train.Country.value_counts())
display(X_test.Country.value_counts())

,count
Country,
United-Kingdom,11
France,11
Germany,5


,count
Country,
United-Kingdom,1
Germany,1
France,1


In [22]:
X_train

,Country,YearsExperience
0,France,1.1
1,United-Kingdom,1.3
2,France,1.5
3,France,2.0
5,United-Kingdom,2.9
6,United-Kingdom,3.0
10,United-Kingdom,3.9
11,United-Kingdom,4.0
12,France,4.0
13,France,4.1


In [29]:
## Sklearn pipeline to preprocess our daza

feature_encoder = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(), ['Country']),
        ('num', MinMaxScaler(), ['YearsExperience'])
    ]
)

feature_encoder = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(), ['Country']),
        ('num', MinMaxScaler(), ['YearsExperience'])
    ]
)

X_train = feature_encoder.fit_transform(X_train)
X_test = feature_encoder.transform(X_test)

In [ ]:
def model_ML(X,y, encoder_strategy, scaler_strategy, model):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
  feature_encoder = ColumnTransformer(
    transformers = [
        ('cat', encoder_strategy, ['Country']),
        ('num', scaler_strategy, ['YearsExperience'])
    ]
)
  X_train = feature_encoder.fit_transform(X_train) ## drop columns
  X_test = feature_encoder.transform(X_test)
  return r2_score(y_train, predictions_train_Set, y_test, predictions_test_Set)

In [24]:
X_train

array([[1.        , 0.        , 0.        ],
       [0.        , 1.        , 0.0212766 ],
       [1.        , 0.        , 0.04255319],
       [1.        , 0.        , 0.09574468],
       [0.        , 1.        , 0.19148936],
       [0.        , 1.        , 0.20212766],
       [0.        , 1.        , 0.29787234],
       [0.        , 1.        , 0.30851064],
       [1.        , 0.        , 0.30851064],
       [1.        , 0.        , 0.31914894],
       [1.        , 0.        , 0.36170213],
       [0.        , 1.        , 0.42553191],
       [0.        , 1.        , 0.44680851],
       [1.        , 0.        , 0.5106383 ],
       [0.        , 1.        , 0.5212766 ],
       [0.        , 1.        , 0.60638298],
       [0.        , 1.        , 0.63829787],
       [1.        , 0.        , 0.72340426],
       [1.        , 0.        , 0.75531915],
       [1.        , 0.        , 0.80851064],
       [0.        , 1.        , 0.89361702],
       [0.        , 1.        , 0.90425532],
       [1.

ValueError: Found unknown categories ['Germany'] in column 0 during transform

In [ ]:
### summary:

## the dummification of the variable Country should have a strategy such as:
## {"UK": 0, "France":1} and by default Germany position is determined by the other 2.

## Point: if the train test split is done in a way that the training pool never saw data from Germany, then
## the strategy would look like {"UK": 0} and then France is determined by what happens in UK feature (0 UK --> 1 France and vice versa).
## However Germany is lost of the picture because the data from it is all in the test set and now we cannot follow.
## This is a edge case that we data scientists need to avoid.
##Why: because the training pool should be as rich as possible (in practice it should contain all the modalities of the categorical data)

In [30]:
regressor = LinearRegression()
regressor.fit(X_train, y_train)

LinearRegression()

In [31]:
predictions_train_Set = regressor.predict(X_train)
predictions_test_Set = regressor.predict(X_test)

In [32]:
r2_score(y_train, predictions_train_Set), r2_score(y_test, predictions_test_Set)

(0.9653917975646743, 0.8469915445654467)

In [33]:
X_train

array([[0.        , 0.        , 1.        , 0.44680851],
       [0.        , 1.        , 0.        , 0.22340426],
       [0.        , 1.        , 0.        , 0.27659574],
       [1.        , 0.        , 0.        , 0.9787234 ],
       [1.        , 0.        , 0.        , 0.80851064],
       [1.        , 0.        , 0.        , 0.30851064],
       [1.        , 0.        , 0.        , 0.        ],
       [0.        , 1.        , 0.        , 0.11702128],
       [0.        , 0.        , 1.        , 0.42553191],
       [0.        , 0.        , 1.        , 0.19148936],
       [1.        , 0.        , 0.        , 0.31914894],
       [0.        , 0.        , 1.        , 0.30851064],
       [1.        , 0.        , 0.        , 0.72340426],
       [0.        , 0.        , 1.        , 0.0212766 ],
       [1.        , 0.        , 0.        , 0.04255319],
       [0.        , 1.        , 0.        , 0.84042553],
       [1.        , 0.        , 0.        , 0.09574468],
       [0.        , 0.        ,

In [34]:
X_test

array([[0.        , 0.        , 1.        , 0.90425532],
       [0.        , 1.        , 0.        , 0.40425532],
       [1.        , 0.        , 0.        , 0.75531915]])